In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 137.9 MB/s eta 0:00:00


In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU (CUDA is not available)")

GPU:  NVIDIA A100-SXM4-80GB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3.8-27B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan cihaz: {device}")

# Fix: Load the correct tokenizer matching the Qwen model instead of gpt2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).to(device)
model = model.eval()

print(f"Parametre sayısı: {model.num_parameters()}")

Kullanılan cihaz: cuda


config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/8.95k [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

Parametre sayısı: 26895998464


In [ ]:
sample= "Yaoay zeka dünyayı değiştiryor."
tokens = tokenizer.tokenize(sample)
ids = tokenizer.encode(sample)
print(f"Token Listesi: {tokens}")
print(f"Token ID'leri: {ids}")
print(f"Geri çözülmüş:", tokenizer.decode(ids))



Token Listesi: ['Y', 'ao', 'ay', 'Ġz', 'eka', 'ĠdÃ¼ny', 'ayÄ±', 'ĠdeÄŁi', 'ÅŁt', 'iry', 'or', '.']
Token ID'leri: [56, 3331, 347, 1110, 50557, 162837, 159601, 215899, 45818, 32652, 269, 13]
Geri çözülmüş: Yaoay zeka dünyayı değiştiryor.


In [ ]:
def ask(question, system="Sen yardımsever bir asistansın. Türkçe kısa cevap ver.", max_new_token=200, temp=0.7, top_p=0.9):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question}
    ]

    # Garanti altına almak için Qwen tokenizer'ını burada doğrudan model_name ile çağırıyoruz
    from transformers import AutoTokenizer
    qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B")

    text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = qwen_tokenizer(text, return_tensors="pt").to(device)

    if qwen_tokenizer.pad_token_id is None:
        qwen_tokenizer.pad_token_id = qwen_tokenizer.eos_token_id

    with torch.no_grad():
        output = model.generate(
            **tokens,
            max_new_tokens=max_new_token,
            do_sample=temp > 0,
            temperature=temp if temp > 0 else None,
            top_p=top_p if temp > 0 else None,
            pad_token_id=qwen_tokenizer.pad_token_id,
        )

    new = output[0][tokens["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new, skip_special_tokens=True)

print(ask("Yapay zeka nedir?"))

We need answer user's simple Turkish question: "Yapay zeka nedir?" Must respond in Turkish short. Need final only.
</think>

Yapay zeka, insan zekâsını taklit ederek öğrenme, akıl yürütme, karar verme ve problem çözme yapabilen bilgisayar sistemleridir.


In [ ]:
questions = [
    "Öklid teoremini açıkla",
    "Jurassic Park'ın yönetmeni kim?",
    "Kürk Mantolu Madonna kitabının özetini ver"
]
for q in questions :
  print(f"Question: {q}\n")
  print(f"Answer: {ask(q)}\n")

Question: Öklid teoremini açıkla

Answer: Kullanıcı Türkçe olarak "Öklid teoremini açıkla" diyor. Kısa Türkçe cevap isteniyor. Öklid teoremi belirsiz olabilir: Öklid geometrisi, Öklid'in beşinci postulatı, Öklid algoritması, Öklid teoremi (sonsuz asal sayı) veya Öklid teoremi (her asal sayı 4k+1 biçimindeyse iki kare toplamı). En yaygın "Öklid teoremi" matematikte "sonsuz sayıda asal sayı vardır" kanıtıdır. Kısa açıkla: varsayım, çarpım, Öklid algoritması/bölen. Türkçe kısa cevap ver.
</think>

Öklid teoremi, genellikle **sonsuz sayıda asal sayı vardır** teoremidir.

Kanıt fikri:  
Varsayalım asal sayılar sonlu olsun: \(

Question: Jurassic Park'ın yönetmeni kim?

Answer: Thinking Process:

1.  **Identify the core entity and question:** The user is asking (in Turkish) for the director of the movie "Jurassic Park".
2.  **Retrieve knowledge:**
    *   Entity: Jurassic Park (1993 film)
    *   Attribute: Director
    *   Fact: Steven Spielberg
3.  **Format the output:** The user requested

In [ ]:
question= "Renkler ile ilgili şiir yaz"
for t in [0,0.7,1.3]:
  print(f"temp= {t}: {ask(question, temp =t, max_new_token=100)}")

temp= 0: We need to respond to user: "Renkler ile ilgili şiir yaz" Turkish: write a poem about colors. Need final in Turkish, short? User asks poem. We can produce a short poem. Ensure final only poem maybe.
</think>

Kırmızı bir gül açar sabahın ucunda,
Sarı güneş fısıldar toprağa.
Mavi gökyüzü uzanır sonsuza,
Yeşil yapraklar dans
temp= 0.7: Kullanıcı Türkçe kısa cevap istiyor ve "Renkler ile ilgili şiir yaz" diyor. Türkçe kısa bir şiir yazmalıyım. Kısa tut, şiir formatında.
</think>

**Renkler**

Kırmızı yanar sabahın dudağında,  
Sarı gülümser güneşin dilinde.  
Mavi kaybolur uzakta, derinde,  
Yeşil fısıldar toprağın dil
temp= 1.3: We need answer in Turkish short due system: "Sen yardımsever bir asistansın. Türkçe kısa cevap ver." User asks "Renkler ile ilgili şiir yaz". Need produce short Turkish poem about colors. Need final only. Ensure short. Maybe 8 lines.
</think>

Her renk bir fısıltıdır,  
Kırmızı tutkuyu söyler,  
Mavi sonsuzluk taşır,  
Sarı güneşle gülümser.




In [ ]:
class Conversation:
  def __init__(self, system = "Sen yardımsever bir asistansın. Türkçe kısa cevap ver."):
    self.past=[{"role":"system","content": system}]

  def message(self, user_message, max_new_token=200):
    self.past.append({"role":"user", "content":user_message})
    text= tokenizer.apply_chat_template(self.past , tokenize=False , add_generation_prompt=True)
    tokens = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
      output = model.generate(
          **tokens,
          max_new_tokens=max_new_token,
          do_sample=True,
          temperature=0.2,
          top_p=0.9,
          pad_token_id=tokenizer.pad_token_id
      )
    answer= tokenizer.decode(output[0][tokens["input_ids"].shape[1]:], skip_special_tokens=True)
    self.past.append({"role":"assistant", "content":answer})
    return answer

conversation = Conversation()
print(conversation.message("Merhaba , Benim adım Özgür ve benim 2 kedim var"))
print(conversation.message("Adımı ve neyim olduğunu hatırlıyor musun?"))


We need to respond to user in Turkish, short. User: "Merhaba , Benim adım Özgür ve benim 2 kedim var". Need answer friendly, maybe ask names. Must be short. Final only Turkish short.
</think>

Merhaba Özgür! Kedilerin çok şanslı. Adları ne?
We need answer in Turkish short. User asks: "Adımı ve neyim olduğunu hatırlıyor musun?" Context: They said name Özgür and has 2 cats. "neyim olduğunu" maybe what am I? They didn't state profession, only has 2 cats. Need answer short: Evet, adın Özgür ve 2 kedisi var. Maybe if asking what I am? "Ben bir asistanım"? But phrase: "Adımı ve neyim olduğunu hatırlıyor musun?" = Do you remember my name and what I am? They only said name and has 2 cats, not what they are. Could say: "Adın Özgür, 2 kedisi var; ne olduğunu söylememiştin." Need short. Ensure Turkish.
</think>

Adın Özgür, 2 kedisi var; ne olduğunu söylememiştin.


In [ ]:
!pip install -q -U transformers peft accelerate

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(model_name)

try:
    # Load the squad v2 adapter directly using the PEFT wrapper
    model = PeftModel.from_pretrained(base_model, "AdapterHub/bert-base-uncased-pf-squad_v2")
    print("Adapter successfully loaded using PEFT!")
except Exception as e:
    print(f"Could not load via PEFT: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adapters 1.1.1 requires transformers~=4.48.3, but you have transformers 5.20.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


adapter_config.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

Could not load via PEFT: 'peft_type'


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

class TurkceBertConversation:
    def __init__(self):
        # Türkçe ve diğer birçok dilde mükemmel sonuç veren açık kaynaklı XLM-RoBERTa SQuAD2 modeli
        model_name = "deepset/xlm-roberta-base-squad2"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.context_memory = ""

    def add_fact(self, fact):
        self.context_memory += fact + " "
        return "Bilgi hafızaya başarıyla eklendi!"

    def ask_question(self, question):
        if not self.context_memory.strip():
            return "Hafızamda henüz hiçbir bilgi yok."

        # Çok dilli model ile Türkçe sorularımızı ve hafızamızı doğrudan işleyebiliriz
        inputs = self.tokenizer(question, self.context_memory, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        answer_start = torch.argmax(outputs.start_logits)
        answer_end = torch.argmax(outputs.end_logits) + 1

        # Token'ları düzgün bir şekilde birleştirip tam metin cevabı elde ediyoruz
        answer_tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end])
        answer = self.tokenizer.convert_tokens_to_string(answer_tokens)
        return answer.strip()

# Yeni modelimizi başlatalım
turkce_model_sohbet = TurkceBertConversation()
turkce_model_sohbet.add_fact("Benim adım Özgür.")
turkce_model_sohbet.add_fact("Benim iki tane kedim var.")
turkce_model_sohbet.add_fact("Ben yazılım mühendisiyim.")
turkce_model_sohbet.add_fact("Ben Fenerbahçeliyim.")

print("Hafıza Güncellendi! Soruları soruyoruz:\n")

# Sorularımızı Türkçe soralım ve sonuçları görelim:
sorular = [
    "Benim adım ne?",
    "Kaç tane kedim var?",
    "Benim mesleğim ne?",
    "Ben hangi takımlıyım?"
]

for soru in sorular:
    cevap = turkce_model_sohbet.ask_question(soru)
    print(f"Soru: {soru} -> Cevap: {cevap}")

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Hafıza Güncellendi! Soruları soruyoruz:

Soru: Benim adım ne? -> Cevap: Özgür
Soru: Kaç tane kedim var? -> Cevap: iki
Soru: Benim mesleğim ne? -> Cevap: yazılım mühendisi
Soru: Ben hangi takımlıyım? -> Cevap: Fenerbahçe
